# 02 — Interaction Edgelist (Full Scale)

Loads the full GloBI dataset and builds the plant-pollinator interaction edgelist
for the complete ANTHEIA pipeline.

**Filtering:**
- CONUS bounding box (24–49.5°N, 125–66°W)
- 7 broad flower-visitation interaction types
- Deduplicated to unique (plant, pollinator) species pairs

**Output:** `globiinteractions_conus.csv` — 715,215 CONUS records; 139 positive pairs
in the shared pair universe after species coverage intersection.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
BASE        = Path("/scratch/ariana.l")
GLOBIPATH   = BASE / "CfE2026CVforEcology" / "rawpollinatordata" / "interactions.csv.gz"
OUT_DIR     = BASE / "New Stage 4 Link Prediction Model"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GLOBIOUT    = OUT_DIR / "globiinteractions_conus.csv"

# ── Constants ───────────────────────────────────────────────────────────────
LAT_MIN, LAT_MAX = 24.0, 49.5
LON_MIN, LON_MAX = -125.0, -66.0

INTERACTION_TYPES = [
    "visits flowers of",
    "visited by",
    "pollinates",
    "pollinated by",
    "has flower-visiting",
    "flower-visiting",
    "interacts with",
]

print("Paths OK")

In [ ]:
# Load full GloBI interactions
print("Loading GloBI interactions...")
df = pd.read_csv(
    GLOBIPATH,
    usecols=["sourceTaxonName", "targetTaxonName", "interactionTypeName",
             "decimalLatitude", "decimalLongitude"],
    low_memory=False
)
print(f"  Total records: {len(df):,}")
df.head()

In [ ]:
# Rename columns
df = df.rename(columns={
    "sourceTaxonName": "plant_species",
    "targetTaxonName": "pollinator_species",
    "interactionTypeName": "interaction_type",
    "decimalLatitude": "lat",
    "decimalLongitude": "lon"
})

# Filter to CONUS and flower-visitation types
df = df[
    (df["interaction_type"].isin(INTERACTION_TYPES)) &
    (df["lat"] >= LAT_MIN) & (df["lat"] <= LAT_MAX) &
    (df["lon"] >= LON_MIN) & (df["lon"] <= LON_MAX)
].dropna(subset=["plant_species", "pollinator_species"])

print(f"  After CONUS + interaction type filter: {len(df):,} records")
print(f"  Unique plant species: {df['plant_species'].nunique():,}")
print(f"  Unique pollinator species: {df['pollinator_species'].nunique():,}")

In [ ]:
# Remove self-paired artifacts
df = df[df["plant_species"] != df["pollinator_species"]]
print(f"  After removing self-pairs: {len(df):,} records")

# Save full filtered records
df.to_csv(GLOBIOUT, index=False)
print(f"  Saved → {GLOBIOUT}")

In [ ]:
# Summary of unique pairs
pairs = df[["plant_species", "pollinator_species"]].drop_duplicates()
print(f"Unique (plant, pollinator) pairs: {len(pairs):,}")
print()
print("Note: the 139 positive pairs used in ANTHEIA are the subset of these")
print("that fall within the shared species coverage after intersecting with")
print("the plant existence matrix F, pollinator existence matrix P,")
print("f_curves, and a_curves. That intersection is computed in the model notebooks.")